In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv("/kaggle/input/ai-for-sustainable-agriculture-dataset/farmer_advisor_dataset.csv")
df.head()

In [ ]:
df1 = pd.read_csv("/kaggle/input/ai-for-sustainable-agriculture-dataset/market_researcher_dataset.csv")
df1.head()

In [ ]:
df.columns

In [ ]:
df1.columns

In [ ]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)
df1.drop_duplicates(inplace=True)

In [ ]:
# Reduce dataset size for faster execution
df = df.sample(frac=0.5, random_state=42)
df1 = df1.sample(frac=0.5, random_state=42)

In [ ]:
# Optimize data types
def optimize_df(df):
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = df[col].astype('int32')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')
    return df

df = optimize_df(df)
df1 = optimize_df(df1)

# Merge datasets

In [ ]:
df_merged = pd.merge(df, df1, left_on='Crop_Type', right_on='Product', how='inner')
df_merged.drop(columns=['Farm_ID', 'Market_ID', 'Product'], inplace=True)

In [ ]:
df_merged.shape

# Data Sampling

In [ ]:
df_merged = df_merged.sample(frac=0.5, random_state=42)  # Keep 50% of data

In [ ]:
df_merged.shape

# EDA

In [ ]:
df_merged.info()

In [ ]:
df_merged.describe()

In [ ]:
df_merged.isnull().sum()

In [ ]:
df_merged.shape

# Data Distribution

In [ ]:
plt.figure(figsize=(12, 6))
df_merged.hist(figsize=(15, 12), bins=20, edgecolor='black')
plt.suptitle("Feature Distributions")
plt.show()

# Outliers detection

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_merged)
plt.xticks(rotation=90)
plt.title("Boxplot for Outlier Detection")
plt.show()

In [ ]:
# Encoding categorical variables
label_encoders = {}
for column in df_merged.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_merged[column] = le.fit_transform(df_merged[column])
    label_encoders[column] = le

# Feature-target split

In [ ]:
y = df_merged['Crop_Yield_ton']  # Target variable
X = df_merged.drop(columns=['Crop_Yield_ton'])

# Model Training

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Ensure all data is numeric before scaling
X_train = X_train.select_dtypes(include=[np.number])
X_test = X_test.select_dtypes(include=[np.number])

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
model = RandomForestRegressor(n_estimators=30, max_depth=7, min_samples_split=5, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Model Evaluation

In [ ]:
print("Mean Absolute Error:", mean_absolute_error(y_test, y_pred))
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("R-squared Score:", r2_score(y_test, y_pred))